In [ ]:
import pandas as pd
import re
import os, sys
tamil = pd.read_csv("tamil.txt", sep="\t", encoding="latin1")
english = pd.read_csv("english.txt", sep="\t", encoding="latin1")
tanglish = pd.read_csv("tanglish.txt", sep="\t", encoding="latin1")


print("Files loaded successfully")

df = pd.concat([tamil, english, tanglish], ignore_index=True)
print("Total rows:", len(df))
df.head()

print(df.columns)

df = df.iloc[:, :2]  # first 2 columns mattum eduthukkum
df.columns = ["comment", "label"]
df.head()

df = df.drop_duplicates(subset="comment")
print("After duplicate removal:", len(df))

df = df.dropna()
df = df[df["comment"].str.strip() != ""]

import re

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+", "", text)          # remove URLs
    text = re.sub(r"@\w+", "", text)             # remove mentions
    text = re.sub(r"#\w+", "", text)             # remove hashtags
    text = re.sub(r"[^a-zA-Z0-9\u0B80-\u0BFF\s]", "", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

print("clean_text function defined")

df["comment"] = df["comment"].apply(clean_text)
df.head()

print(df["label"].unique())
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

df["label_encoded"] = le.fit_transform(df["label"])

df.head()

for label, code in zip(le.classes_, le.transform(le.classes_)):
    print(label, "->", code)

df[["comment", "label", "label_encoded"]].head()

X = df["comment"]
y = df["label_encoded"]

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

print("Train label distribution:")
print(y_train.value_counts())

print("\nTest label distribution:")
print(y_test.value_counts())

from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1,2)
)

X_train_vec = tfidf.fit_transform(X_train)
X_test_vec = tfidf.transform(X_test)

print("Train vector shape:", X_train_vec.shape)
print("Test vector shape:", X_test_vec.shape)

from sklearn.linear_model import LogisticRegression

model = LogisticRegression(
    max_iter=1000
)

model.fit(X_train_vec, y_train)
print("Model training completed ✅")

y_pred = model.predict(X_test_vec)

from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))

import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

ConfusionMatrixDisplay.from_predictions(y_test, y_pred)
plt.show()

model = LogisticRegression(
    max_iter=2000,
    class_weight="balanced"
)

model.fit(X_train_vec, y_train)

y_pred = model.predict(X_test_vec)

from sklearn.metrics import classification_report, accuracy_score

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

def predict_comment(text):
    text = clean_text(text)
    vec = tfidf.transform([text])
    pred = model.predict(vec)[0]
    return le.inverse_transform([pred])[0]

print(predict_comment("no da"))
print(predict_comment("dei romba mokka da"))
print(predict_comment("sethuru da"))


Files loaded successfully
